# Time-to-platinum: vitals × feature-statistic ablation

Fit **XGBoost survival:cox and elastic-net Cox** on the full ADT cohort at **0, 90, and 180 days**.
At every landmark, cross **with / without vitals** with **mean + n_obs + missing / all statistics**:
24 fits, each with five-fold tuning and held-out test predictions.

| Combination | Vitals | Feature statistics |
|---|---|---|
| With vitals / all stats | Included | All statistics + missingness |
| Without vitals / all stats | Excluded | All statistics + missingness |
| With vitals / compact stats | Included | Mean + n_obs + missingness |
| Without vitals / compact stats | Excluded | Mean + n_obs + missingness |

The results show every combination side by side and all six pairwise contrasts within each model/landmark,
including contrasts that change both vitals and statistics, with paired-patient bootstrap 95% CIs on the held-out
metrics. Training wall time is recorded for each new fit.

All arms retain age, the same prebuilt patient splits, landmark risk sets, death-censored platinum outcome,
and evaluation horizons. Follow-up is measured from each landmark. Only predictor candidates change;
coverage/variability selection, imputation, scaling, and missingness indicators are learned within training folds.

- `mean-n-obs`: `mean` + `n_observations` (the stored name for n_obs), with train-derived missing indicators.
- `all`: every stored summary statistic (currently mean, min, max, last, delta, n_observations), with the same missing-indicator logic.
- Vitals: height, weight, temperature, heart rate, respiratory rate, systolic and diastolic blood pressure,
  using the canonical names printed below. Inclusion makes them eligible; training coverage/variability gates still apply.

Requires the prediction inputs from `01_preprocessing.ipynb` and the modeling environment used by `03_multivariate.ipynb`.
Results go into a separate `platinum_feature_ablation` tree. Existing completed fits resume by default.

In [ ]:
import json
import shlex
import subprocess
import sys
import time
from datetime import datetime, timezone
from itertools import combinations, product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Resolve imports whether launched from the repository root or notebook directory.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "COMPASS/survival_analysis/multivariate_analysis.py").exists())
sys.path.insert(0, str(REPO_ROOT))
from COMPASS.survival_analysis.cox_aggregated import ENDPOINTS, ID_COL, VITAL_MEASUREMENTS
from survival_common.heldout_bootstrap import (
    assert_reproduces_stored, bootstrap_heldout, load_test_predictions, percentile_ci,
)
from survival_common.helper import resolve_auc_max_time_units

SURVIVAL_ROOT = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis")
INPUTS_DIR = SURVIVAL_ROOT / "prediction_inputs_adt"
OUTPUT_ROOT = SURVIVAL_ROOT / "local_runs_adt" / "platinum_feature_ablation"
LANDMARKS = [0, 90, 180]
MODELS = ["elastic-net", "xgboost"]
VITAL_SETTINGS = ["with_vitals", "without_vitals"]
STAT_SETTINGS = ["mean-n-obs", "all"]
N_FOLDS = 5
SEED = 42
N_BOOTSTRAP = 1000  # patient resamples of each landmark's test set
OVERWRITE = False
DRY_RUN = False  # print commands without fitting or writing outputs
OUT_OF_FOLD_RISKS = False  # optional nested CV; test-block predictions are always written

TASKS = []
for model, landmark, vitals, stats in product(MODELS, LANDMARKS, VITAL_SETTINGS, STAT_SETTINGS):
    folder = OUTPUT_ROOT / vitals / stats / model / f"landmark_{landmark}"
    prefix = "cox_agg_multivariable" if model == "elastic-net" else "landmark_xgboost"
    cmd = [sys.executable, str(REPO_ROOT / "COMPASS/survival_analysis/multivariate_analysis.py"),
           "--model", model, "--inputs-dir", str(INPUTS_DIR.resolve()), "--output-dir", str(folder.resolve()),
           "--endpoints", "platinum", "--landmark-days", str(landmark), "--feature-set", "labs",
           "--feature-stats", stats, "--n-folds", str(N_FOLDS), "--seed", str(SEED), "--cohort", "all",
           "--save-test-survival", "--overwrite" if OVERWRITE else "--no-overwrite"]
    if vitals == "without_vitals":
        cmd.append("--exclude-vitals")
    if OUT_OF_FOLD_RISKS:
        cmd.append("--out-of-fold-risks")
    TASKS.append(dict(model=model, landmark=landmark, vitals=vitals, stats=stats,
                      folder=folder, metrics=folder / f"{prefix}_metrics_landmark{landmark}.csv",
                      risks=folder / f"{prefix}_patient_risks_landmark{landmark}.csv", command=cmd))
print(f"{len(TASKS)} fits; vitals: {', '.join(sorted(VITAL_MEASUREMENTS))}")
display(pd.DataFrame(TASKS)[["model", "landmark", "vitals", "stats", "folder"]])

## Check inputs

This checks all landmarks before starting any fit and shows which stored measurements/statistics are available.
No patient is excluded for missing vitals. A missing vital panel raises an error so the two vitals arms cannot
silently run on identical feature sets. The source prediction inputs are read directly and never rewritten.

In [ ]:
required = [INPUTS_DIR / "build_manifest.json"]
for lm in LANDMARKS:
    required.extend([INPUTS_DIR / f"aggregated_landmark{lm}.csv",
                     INPUTS_DIR / f"pre_treatment_lab_long_landmark{lm}.csv"])
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Build inputs with notebook 01 or update INPUTS_DIR:\n" + "\n".join(missing))
build_manifest = json.loads((INPUTS_DIR / "build_manifest.json").read_text())
audit = []
for lm in LANDMARKS:
    cols = pd.read_csv(INPUTS_DIR / f"aggregated_landmark{lm}.csv", nrows=0).columns
    features = [c for c in cols if "__" in c]
    names = {c.rsplit("__", 1)[0] for c in features}
    present_vitals = names & VITAL_MEASUREMENTS
    if not present_vitals:
        raise ValueError(f"No canonical vitals found at landmark {lm}; rebuild inputs including vitals.")
    if not build_manifest["auc_horizons_by_landmark"][str(lm)]["platinum"]:
        raise ValueError(f"No platinum evaluation horizons for landmark {lm}.")
    for vitals, stats in product(VITAL_SETTINGS, STAT_SETTINGS):
        selected = [c for c in features
                    if (vitals == "with_vitals" or c.rsplit("__", 1)[0] not in VITAL_MEASUREMENTS)
                    and (stats == "all" or c.rsplit("__", 1)[1] in {"mean", "n_observations"})]
        if not selected:
            raise ValueError(f"No candidates for {lm}, {vitals}, {stats}")
        audit.append(dict(landmark=lm, vitals=vitals, stats=stats, n_candidates=len(selected),
                          stored_stats=sorted({c.rsplit("__", 1)[1] for c in selected}),
                          available_vitals=sorted(present_vitals)))
display(pd.DataFrame(audit))

## Fit all variants

Each task writes the usual metrics, per-patient held-out risk scores and survival probabilities, feature selections,
and coefficients or importance files. `run_settings.json` records the ablation and command. Changing settings in an existing output
folder requires `OVERWRITE = True` or a new `OUTPUT_ROOT`. To refit after changing source data or model code,
set `OVERWRITE = True` as well. Nested out-of-fold predictions are optional because they add substantial tuning work.

`training_time.json` records elapsed wall-clock seconds and UTC start/end timestamps per fit. The timer spans
the model subprocess: input loading, preprocessing, cross-validation/tuning, final fitting, evaluation, artifact
writing, and optional nested out-of-fold work. It measures total training-job cost, not just the final estimator fit.
Resuming preserves the original duration. Older completed runs without a timing record show missing time;
use `OVERWRITE = True` to measure them. Failed/interrupted attempts are marked separately from completed fits.


In [ ]:
for task in TASKS:
    print(shlex.join(task["command"]), flush=True)
    if DRY_RUN:
        continue
    folder = task["folder"]
    settings = {k: task[k] for k in ("model", "landmark", "vitals", "stats")}
    settings.update(endpoint="platinum", inputs_dir=str(INPUTS_DIR.resolve()), n_folds=N_FOLDS,
                    seed=SEED, out_of_fold_risks=OUT_OF_FOLD_RISKS, build_manifest=build_manifest)
    settings_path = folder / "run_settings.json"
    if settings_path.exists() and not OVERWRITE:
        previous = json.loads(settings_path.read_text())
        if previous["settings"] != settings:
            raise ValueError(f"Settings changed under {folder}; use OVERWRITE=True or a new OUTPUT_ROOT.")
    timing_path = folder / "training_time.json"
    if task["metrics"].exists() and not OVERWRITE:
        if timing_path.exists() and json.loads(timing_path.read_text())["status"] != "completed":
            raise ValueError(f"Previous fit did not complete under {folder}; use OVERWRITE=True.")
        if not settings_path.exists():
            raise ValueError(f"Cannot verify existing fit under {folder}; use OVERWRITE=True.")
        print(f"[skip] {task['metrics']}")
        continue
    folder.mkdir(parents=True, exist_ok=True)
    settings_path.write_text(json.dumps({"settings": settings, "command": task["command"]}, indent=2) + "\n")
    timing = dict(status="running", started_at_utc=datetime.now(timezone.utc).isoformat(),
                  finished_at_utc=None, training_seconds=None,
                  scope="full_model_subprocess_including_cv_and_evaluation")
    timing_path.write_text(json.dumps(timing, indent=2) + "\n")
    started = time.perf_counter()
    try:
        subprocess.run(task["command"], cwd=REPO_ROOT, check=True)
    except BaseException as exc:
        timing["status"] = "failed"
        timing["error_type"] = type(exc).__name__
        raise
    else:
        timing["status"] = "completed"
    finally:
        timing["training_seconds"] = time.perf_counter() - started
        timing["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        timing_path.write_text(json.dumps(timing, indent=2) + "\n")
    print(f"[completed] Training job: {timing['training_seconds'] / 60:.2f} min", flush=True)

## Held-out comparison

Compare mean AUC(t), C-index, and integrated Brier on each landmark's test set. Higher AUC/C-index and lower
Brier are better. Different landmarks have different at-risk populations, so changes across landmarks do not
isolate the benefit of accumulating history. The following contrasts hold model and landmark fixed.

The four-combination table includes training minutes alongside prediction metrics. Pairwise deltas below are
**combination A minus combination B**, with the full settings named on both sides. A positive AUC/C-index delta
favors A; a negative Brier delta favors A. Positive training-time deltas mean A took longer. Contrasts that change
both settings measure their combined effect, not an isolated vitals or statistics effect. The bootstrap section below
adds confidence intervals. Timing comparisons also depend on hardware and concurrent workload.


In [ ]:
rows = []
for task in TASKS:
    row = {k: task[k] for k in ("model", "landmark", "vitals", "stats")}
    row.update(status="missing", source_path=str(task["metrics"]), risks_path=str(task["risks"]))
    if task["metrics"].exists():
        frame = pd.read_csv(task["metrics"])
        frame = frame.loc[frame["endpoint"] == "platinum"]
        if len(frame) != 1:
            raise ValueError(f"Expected one platinum metrics row: {task['metrics']}")
        row.update(frame.iloc[0].to_dict())
        row.update({k: task[k] for k in ("model", "landmark", "vitals", "stats")})
        row["status"] = "ok"
    timing_path = task["folder"] / "training_time.json"
    timing = json.loads(timing_path.read_text()) if timing_path.exists() else {}
    row["timing_status"] = timing.get("status", "not_recorded")
    row["training_seconds"] = timing.get("training_seconds", np.nan) if timing.get("status") == "completed" else np.nan
    row["training_minutes"] = row["training_seconds"] / 60
    if timing and timing["status"] != "completed":
        row["status"] = "training_" + timing["status"]
    row["combination"] = ("With vitals" if task["vitals"] == "with_vitals" else "Without vitals") + " / " + (
        "all stats" if task["stats"] == "all" else "mean + n_obs + missing")
    rows.append(row)
summary = pd.DataFrame(rows)
metrics = ["test_mean_auc_t", "test_c_index", "test_integrated_brier"]
for col in [*metrics, "n_test", "n_events_test"]:
    if col not in summary:
        summary[col] = np.nan
display(summary[["model", "landmark", "vitals", "stats", "status", "n_test", "n_events_test", *metrics, "training_minutes", "timing_status"]])
if not DRY_RUN:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    summary.to_csv(OUTPUT_ROOT / "ablation_summary.csv", index=False)

ok = summary.loc[summary["status"] == "ok"]
if not ok.empty:
    counts = ok.groupby("landmark")[["n_test", "n_events_test"]].nunique()
    if (counts > 1).any().any():
        raise ValueError("Test cohort counts differ across variants; inspect inputs and stale outputs before comparing.")
    cross_comparison = ok.pivot(index=["model", "landmark"], columns="combination",
                                values=[*metrics, "training_minutes"])
    display(cross_comparison.round(3))
    if not DRY_RUN:
        cross_comparison.to_csv(OUTPUT_ROOT / "ablation_cross_comparison.csv")

## Bootstrap confidence intervals

Each landmark's test patients are resampled with replacement `N_BOOTSTRAP` times, and every fit at that landmark is
rescored on the same draws. Sharing draws makes each contrast below a paired comparison: its interval comes from the
per-draw difference between two fits on the same patients, not from two intervals side by side.

- **Same metrics:** draws call the functions the fits used, with the same train-side IPCW reference, horizon grid and
  evaluation cap, and support and coverage gates reapplied in each draw. Recomputed values must match each stored
  metric, or this section stops.
- **What the intervals cover:** test-sampling uncertainty only. Fitted models and tuned hyperparameters are held fixed,
  so refitting and tuning variability are not included.
- **Reading them:** intervals are 95% percentile intervals, unadjusted for multiplicity. The six contrasts across six
  model/landmark groups and three metrics give 108 intervals, so about five would exclude zero by chance alone.
- **Integrated Brier:** this needs each patient's S(t|x), which only fits run with `--save-test-survival` store. Older
  fits still get C-index and AUC(t) intervals; their Brier intervals stay blank until refit with `OVERWRITE = True`.


In [ ]:
endpoint = ENDPOINTS["platinum"]
time_unit_days = int(build_manifest["auc_time_unit_days"])
max_time_unit = resolve_auc_max_time_units(build_manifest, None)

draw_frames = []
for landmark, landmark_fits in ok.groupby("landmark", sort=False):
    columns = {endpoint["duration_col"]: "duration_days", endpoint["event_col"]: "event"}
    inputs = pd.read_csv(INPUTS_DIR / f"aggregated_landmark{landmark}.csv", usecols=["split", *columns])
    reference = inputs.loc[inputs["split"].isin(["train", "valid"])].rename(columns=columns)
    horizons = np.asarray(build_manifest["auc_horizons_by_landmark"][str(landmark)]["platinum"], dtype=float)
    # The same seed gives elastic-net and XGBoost the same patients at this landmark.
    for model, group in landmark_fits.groupby("model", sort=False):
        fits = {row["combination"]: load_test_predictions(row["risks_path"]) for _, row in group.iterrows()}
        observed, fit_draw_frame = bootstrap_heldout(
            fits, reference=reference, horizons=horizons, time_unit_days=time_unit_days,
            max_time_unit=max_time_unit, n_boot=N_BOOTSTRAP, seed=SEED + int(landmark), id_col=ID_COL,
        )
        stored = group.set_index("combination")
        for _, row in observed.iterrows():
            assert_reproduces_stored(row, stored.loc[row["fit"]],
                                     context=f"{model} +{landmark}d {row['fit']}")
        draw_frames.append(fit_draw_frame.rename(columns={"fit": "combination"})
                           .assign(model=model, landmark=landmark))
draws = (pd.concat(draw_frames, ignore_index=True) if draw_frames
         else pd.DataFrame(columns=["model", "landmark", "combination", "draw", *metrics]))

ci_rows = []
for (model, landmark, combination), group in draws.groupby(["model", "landmark", "combination"]):
    row = dict(model=model, landmark=landmark, combination=combination)
    for metric in metrics:
        row[f"{metric}_ci_low"], row[f"{metric}_ci_high"], row[f"{metric}_n_valid_draws"] = percentile_ci(group[metric])
    ci_rows.append(row)
ci_columns = [f"{metric}_{part}" for metric in metrics for part in ("ci_low", "ci_high", "n_valid_draws")]
ci_summary = ok[["model", "landmark", "vitals", "stats", "combination", "n_test", "n_events_test", *metrics]].merge(
    pd.DataFrame(ci_rows, columns=["model", "landmark", "combination", *ci_columns]),
    on=["model", "landmark", "combination"], validate="one_to_one",
)


def with_ci(row, metric):
    low, high = row[f"{metric}_ci_low"], row[f"{metric}_ci_high"]
    return f"{row[metric]:.3f} ({low:.3f}, {high:.3f})" if np.isfinite(low) else f"{row[metric]:.3f} (no CI)"


if not ci_summary.empty:
    ci_table = ci_summary.assign(**{metric: ci_summary.apply(with_ci, axis=1, metric=metric) for metric in metrics})
    display(ci_table.pivot(index=["model", "landmark"], columns="combination", values=metrics))
    display(ci_summary[["model", "landmark", "combination", "n_test", "n_events_test"]])
    no_brier = ci_summary["test_integrated_brier_ci_low"].isna().sum()
    if no_brier:
        print(f"{no_brier}/{len(ci_summary)} fits have no integrated Brier CI: "
              "refit them with OVERWRITE = True to save test survival.")
    if not DRY_RUN:
        ci_summary.to_csv(OUTPUT_ROOT / "ablation_bootstrap_ci.csv", index=False)
        draws.to_csv(OUTPUT_ROOT / "ablation_bootstrap_draws.csv", index=False)


In [ ]:
# All six pairwise comparisons among the four combinations, within model/landmark.
comparison_metrics = [*metrics, "training_seconds", "training_minutes"]
contrast_columns = ["model", "landmark", "combination_a", "combination_b",
                    "vitals_a", "stats_a", "vitals_b", "stats_b", "contrast_type",
                    *[f"delta_{metric}" for metric in comparison_metrics],
                    *[f"delta_{metric}_ci_{bound}" for metric in metrics for bound in ("low", "high")]]
fit_draws = {key: group.set_index("draw") for key, group in draws.groupby(["model", "landmark", "combination"])}
contrasts = []
for (model, landmark), group in ok.groupby(["model", "landmark"], sort=False):
    records = group.sort_values(["vitals", "stats"]).to_dict("records")
    for a, b in combinations(records, 2):
        changes = [key for key in ("vitals", "stats") if a[key] != b[key]]
        contrast = dict(model=model, landmark=landmark,
                        combination_a=a["combination"], combination_b=b["combination"],
                        vitals_a=a["vitals"], stats_a=a["stats"],
                        vitals_b=b["vitals"], stats_b=b["stats"],
                        contrast_type=" + ".join(changes))
        for metric in comparison_metrics:
            contrast[f"delta_{metric}"] = a[metric] - b[metric]
        draws_a = fit_draws[(model, landmark, a["combination"])]
        draws_b = fit_draws[(model, landmark, b["combination"])]
        for metric in metrics:
            low, high, _ = percentile_ci(draws_a[metric] - draws_b[metric])
            contrast[f"delta_{metric}_ci_low"] = low
            contrast[f"delta_{metric}_ci_high"] = high
        contrasts.append(contrast)
contrast_df = pd.DataFrame(contrasts, columns=contrast_columns)
display(contrast_df)
if not DRY_RUN:
    contrast_df.to_csv(OUTPUT_ROOT / "ablation_contrasts.csv", index=False)

if not ok.empty:
    fig, axes = plt.subplots(1, len(LANDMARKS), figsize=(6 * len(LANDMARKS), 5), sharey=True, squeeze=False)
    for ax, lm in zip(axes[0], LANDMARKS):
        sub = ci_summary.loc[ci_summary["landmark"] == lm].copy()
        sub["label"] = sub["model"] + " | " + sub["vitals"] + " | " + sub["stats"]
        # A percentile interval need not contain the point estimate.
        xerr = [(sub["test_mean_auc_t"] - sub["test_mean_auc_t_ci_low"]).clip(lower=0),
                (sub["test_mean_auc_t_ci_high"] - sub["test_mean_auc_t"]).clip(lower=0)]
        ax.barh(sub["label"], sub["test_mean_auc_t"], xerr=xerr, capsize=3)
        ax.axvline(0.5, color="black", ls=":", lw=0.7)
        ax.set_xlim(0, 1)
        ax.set_title(f"Landmark {lm} days")
        ax.set_xlabel("Test mean AUC(t) (bootstrap 95% CI)")
    fig.tight_layout()
    if not DRY_RUN:
        fig.savefig(OUTPUT_ROOT / "ablation_mean_auc.png", dpi=180, bbox_inches="tight")
    plt.show()